# Simplified Chinese support and reach on Steam

**DATA2010 Data Science Programming · final project**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theonlypgk/data2010-steam-chinese/blob/main/steam_chinese_analysis.ipynb)

**Main question:** should a Steam game support Simplified Chinese, for which genres, and when?
Reach = number of reviews. This notebook is the whole project, from the two raw input files
to the answers. Run it from top to bottom and it rebuilds every output table.

| Part | What it does | Output tables |
| :--- | :--- | :--- |
| [Setup](#scrollTo=setup) | connect the runtime to the project folder and check the package versions | |
| [Step 1 · Clean](#scrollTo=step1) | load the two snapshots, clean them, filter and encode | `pcd_games_clean` |
| [Step 2 · SQ1 groups](#scrollTo=step2) | group similar games, with vs without Chinese; test the Chinese-game filter | `pcd_sq1_groups` |
| [Step 3 · SQ2 groups](#scrollTo=step3) | group similar games, added Chinese after launch vs did not | `pcd_sq2_groups` |
| [Save and check](#scrollTo=save) | write the three tables, stop with an error if anything is wrong | |
| [Part B · Analysis](#scrollTo=partb) | figures and insights for EDA, SQ1, SQ2, SQ3 (reads the saved tables only) | `report/figures/` |

Each table is saved whole as `data/processed/csv/pcd_<table>.csv` (to read in code) and
`data/processed/xlsx/pcd_<table>.xlsx` (to open and look at).

### How to run

- **Google Colab:** click *Open in Colab*, then *Runtime → Run all*. Setup clones the
  repository into the runtime and installs the pinned package versions. On a new runtime it
  stops once and asks for *Runtime → Restart session*; after that, *Run all* runs to the end.
  Files written on Colab stay in the runtime and disappear when it is deleted.
- **Own computer or Codespaces:** Python 3.12 or 3.13, `pip install -r requirements.txt`, open
  the notebook from the repository folder and choose *Run All*, or run
  `jupyter nbconvert --to notebook --execute steam_chinese_analysis.ipynb --inplace`.

**Rules for a reproducible run**
- Run the cells in order, from Setup down. To repeat any part, start again from Setup.
- The notebook is the only thing that writes the files in `data/processed/`; they are never
  edited by hand.
- Every run gives the same output tables: the inputs are checked against their SHA-256
  fingerprints and the package versions are pinned. The CSV files are byte-identical on every
  machine; the xlsx files hold identical content (only their zip compression depends on the
  machine's zlib library).

**How to read it.** The markdown at the top of each step explains the strategy and why each
choice was made. Each code cell holds one decision: it starts with its label (1.2a, 1.2b, ...),
the decision and the reason for it. Each function has a comment block saying what it does
and why; each fixed value is set in the cell that first uses it. Problems found in
the real data are marked `DATA ISSUE`. The **Check** notes after a result give the numbers a
correct run shows; the checks in *Save and check* stop the notebook if any of them differs.

## Setup · Connect the runtime to the project

Run cells a-c first, on any machine.

- **Project folder.** The notebook looks for the repository folder from where it is opened.
  On Colab, which starts in an empty `/content` folder, it clones the repository there first.
  It then moves into the repository folder, so every path below means the same thing on every
  machine.
- **Package versions.** The installed versions are compared with `requirements.txt`. The same
  code can give different results with a different pandas or numpy version, so a mismatch
  stops the notebook; on Colab the pinned versions are installed instead. `jupyter` and
  `streamlit` are not checked: they open the notebook and the app, the analysis does not use
  them.

In [ ]:
# Setup a · Find the project folder
# Why: the notebook must run the same way wherever it is opened: from the repository folder,
# from a subfolder, or on Colab, which starts in an empty folder and needs the repository
# cloned first. Every later path is relative to the folder found here. The Python versions
# are the ones on which the pinned packages were run end to end.

import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/theonlypgk/data2010-steam-chinese"
IN_COLAB = "google.colab" in sys.modules           # True when the notebook runs on Google Colab
TESTED_PYTHON = {(3, 12), (3, 13)}                  # versions on which the pins were run end to end

if sys.version_info[:2] not in TESTED_PYTHON:
    raise RuntimeError(f"Python {sys.version.split()[0]}: use Python 3.12 or 3.13 (requirements.txt pins).")


# find_root: locate the repository folder.
#   The notebook may be opened from the repository folder, from a subfolder, or (on Colab)
#   from no folder at all. Walking up from the current folder finds the repository in the
#   first two cases; both files must be there, so an unrelated folder is never taken.
#   Returns the folder, or None when the notebook is not inside the repository.
def find_root():
    """First folder, from here upwards, that holds requirements.txt and this notebook."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "requirements.txt").exists() and (folder / "steam_chinese_analysis.ipynb").exists():
            return folder
    return None


ROOT = find_root()
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/data2010-steam-chinese")
    if not ROOT.exists():
        # GIT_LFS_SKIP_SMUDGE=1: leave the two large input files out of the clone;
        # Step 1.1 fetches them from the release, which has no Git LFS download quota.
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True,
                       env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
if ROOT is None:
    raise FileNotFoundError(f"Open the notebook inside the repository folder (git clone {REPO_URL}).")
os.chdir(ROOT)                                      # every relative path now starts here
print(f"Python {sys.version.split()[0]} | {'Google Colab' if IN_COLAB else 'local'} | project: {ROOT}")

In [ ]:
# Setup b · Use exactly the package versions of requirements.txt
# Why: the same code can give different numbers with a different pandas or numpy version, so a
# mismatch stops the notebook instead of silently producing other results. On Colab the
# pinned versions are installed. jupyter and streamlit are not checked: they open the notebook
# and the app, the analysis does not import them.

NOT_NEEDED_TO_RUN = {"jupyter", "streamlit"}        # open the notebook / the app; not imported here


# pinned_versions: read the exact versions the project was run with.
#   requirements.txt is the single place where versions are fixed, so the check below reads
#   it instead of repeating the numbers here. Comments after "#" are ignored; jupyter and
#   streamlit are skipped because the analysis does not import them.
#   Returns {"pandas": "3.0.5", ...}.
def pinned_versions() -> dict:
    """Package -> exact version from requirements.txt, for the packages this notebook imports."""
    pins = {}
    for line in (ROOT / "requirements.txt").read_text().splitlines():
        line = line.split("#")[0].strip()          # drop the comments
        if "==" in line:
            name, version = (part.strip() for part in line.split("=="))
            if name not in NOT_NEEDED_TO_RUN:
                pins[name] = version
    return pins


# installed_version: which version of a package this machine has.
#   importlib.metadata reads the installed package's metadata without importing it, so the
#   check does not load an old version into memory.
#   Returns the version text, or None if the package is not installed.
def installed_version(name: str):
    """Installed version of a package, or None if it is not installed."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


PINS = pinned_versions()
wrong = {name: pin for name, pin in PINS.items() if installed_version(name) != pin}
if wrong and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[f"{name}=={pin}" for name, pin in wrong.items()]], check=True)
    importlib.invalidate_caches()
    # A package that was already imported keeps its old version until the session restarts.
    loaded = [name for name in wrong if name.replace("-", "_") in sys.modules]
    if loaded:
        raise RuntimeError(f"Installed the pinned versions of {', '.join(loaded)}. "
                           "Choose Runtime > Restart session, then Runtime > Run all.")
    wrong = {name: pin for name, pin in PINS.items() if installed_version(name) != pin}
if wrong:
    found = {name: installed_version(name) for name in wrong}
    raise RuntimeError(f"Versions differ from requirements.txt: {found}. "
                       "Run pip install -r requirements.txt, restart the kernel, then Run All.")

print(" | ".join(f"{name} {pin}" for name, pin in PINS.items()))

In [ ]:
# Setup c · Import the libraries and name the project folders
# Why: every library is imported once, here, so the cells below only hold analysis code; the
# folder names are the ones the README describes.

import ast                    # literal_eval: read "['English', 'French']" as a Python list
import csv                    # read the raw header row on its own
import gzip                   # unpack the release copy of an input file
import hashlib                # SHA-256 fingerprints of the input and output files
import re                     # regular expressions for text cleaning and name matching
import shutil                 # copy files without loading them into memory
import urllib.request         # download the release copy of an input file
import zipfile                # rewrite the xlsx files with fixed dates
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

RAW = ROOT / "data" / "raw"                           # the two input files (Git LFS)
PROCESSED_CSV = ROOT / "data" / "processed" / "csv"   # output tables for code
PROCESSED_XLSX = ROOT / "data" / "processed" / "xlsx" # same tables, to open and look at
PREFIX = "pcd_"                                       # file prefix of every output table
print("imports done; raw inputs in", RAW)

# Step 1 · Clean the data and encode it → `pcd_games_clean`

**Strategy.** Load the two raw snapshots, clean both in exactly the same way (so a game
looks the same in both), remove the rows that cannot answer the question, and encode the
binned columns. The result for snapshot A is the cleaned dataset `pcd_games_clean`, one row
per game; the cleaned snapshot C is used again in Step 3.

| Sub-step | What | Why |
|---|---|---|
| 1.1 Load | check A and C against their SHA-256 fingerprints (fetch them if missing), keep 13 of 40 columns, English names | the same input bytes on every run |
| 1.2 Clean | fix types and data issues, parse list cells, add outcome, treatment, confounders and filter flags; encode the bins | below |
| 1.3 Filter | remove Chinese-first games, Early Access, < 10 reviews, < 1 year old, no genre/price/year | below |
| 1.4 Build | keep the columns of the cleaned dataset | |
| 1.5 Check | show how many games each filter removed, which Chinese-first games were removed, and a random sample checked by hand | to check the rules |

## 1.1 Load the two raw snapshots

Snapshot A (2026-06-20, the last version of `games.csv`; later commits only update a 1 GB
`games.json`) and snapshot C (2024-01-11), from the Hugging Face dataset
`FronkonGames/steam-games-dataset`, pinned by commit hash so the data never changes even
though the author keeps updating it.

**Same input bytes on every machine.** Each file is accepted only if its SHA-256 fingerprint
matches the one below; a single changed byte gives a different fingerprint. The notebook takes
the first copy that passes, in this order:

| Copy | When it is used |
| :--- | :--- |
| `data/raw/games_<a/c>.csv` | committed with Git LFS; present after a normal `git clone` |
| GitHub release `raw-snapshots-v1` (gzip) | Colab, or a clone made without Git LFS: there the two names hold 134-byte pointer files, which the fingerprint check rejects instead of failing later with shifted columns |
| Hugging Face, pinned commit | if the release cannot be reached |

**DATA ISSUE.** The header of snapshot A merges two names (`DiscountDLC count`) while every
row holds two values; without a fix every later column is shifted by one. The header is
repaired before reading, and the load stops with an error if the columns still look shifted.

In [ ]:
# 1.1a · Pin the input data and its fingerprints
# Why: the dataset on Hugging Face keeps being updated, so it is read at fixed commits; and a
# file is accepted only if its SHA-256 fingerprint matches, so every run starts from exactly
# the same bytes.

# Data source: two versions of the same file, pinned by commit hash.
HF_DATASET = "FronkonGames/steam-games-dataset"
HF_FILENAME = "games.csv"
REVISION = {
    "a": "52ba62abfa707a2fa31f4d147c2777a9d9fbd2da",   # snapshot A: last update of games.csv
    "c": "aefdeb1ff1a739b92ac5338682cc2610a0328505",   # snapshot C: January 2024 catalogue
}
SNAPSHOT_DATE_A = "2026-06-20"                          # day snapshot A was taken
SNAPSHOT_DATE_C = "2024-01-11"                          # day snapshot C was taken
# SHA-256 of each raw file; the same values are in SHA256SUMS.txt of the release.
SHA256 = {
    "a": "1b48008b01a799d82385d6e66ba0cb65dd477b6aa4b6c2ec8c443025ab6880ad",
    "c": "c755572b804a5c43f4f005aacef23cadfe92ae77d03daebf908d521c81285821",
}
RELEASE_URL = REPO_URL + "/releases/download/raw-snapshots-v1/games_{}.csv.gz"


# sha256_of: fingerprint a file.
#   SHA-256 turns a file into a 64-character code; changing a single byte changes the code
#   completely. Comparing it with the stored value proves the file is exactly the one the
#   results were produced from. The file is read in 1 MB blocks so a 400 MB file never has
#   to fit in memory at once.
def sha256_of(path) -> str:
    """SHA-256 fingerprint of a file, read in 1 MB blocks."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):   # 1 << 20 bytes = 1 MB; stops at b"" (end)
            digest.update(block)                          # add each block to the fingerprint
    return digest.hexdigest()

In [ ]:
# 1.1b · Fetch an input file that is missing or wrong
# Why: data/raw holds the right files after a normal clone, but not on Colab or in a clone
# without Git LFS (there the names hold small pointer files). The release copy is tried first
# because it has no download quota; Hugging Face at the pinned commit is the last resort.
# Whatever the source, the fingerprint is checked again.

# from_release: first fallback source of an input file.
#   The GitHub release raw-snapshots-v1 holds gzip copies of both files (about 3x smaller
#   than the csv). Release downloads, unlike Git LFS, have no monthly download quota, so
#   every Colab run can use them. The .gz file is unpacked to the csv and then deleted.
def from_release(label: str, target) -> None:
    """Download the gzip copy of one snapshot from the GitHub release and unpack it."""
    packed = target.with_suffix(".csv.gz")                  # games_a.csv -> games_a.csv.gz
    urllib.request.urlretrieve(RELEASE_URL.format(label), packed)   # download the gzip copy
    with gzip.open(packed, "rb") as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)                       # unpack without loading it into memory
    packed.unlink()                                        # delete the .gz, keep the csv


# from_hugging_face: last fallback source of an input file.
#   The original dataset on Hugging Face, read at the pinned commit (REVISION), so it is
#   the same version as the release copy. huggingface_hub is imported here, not at the
#   top, because most runs never reach this function.
def from_hugging_face(label: str, target) -> None:
    """Download games.csv of one snapshot from Hugging Face at its pinned commit."""
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(repo_id=HF_DATASET, filename=HF_FILENAME,
                             revision=REVISION[label], repo_type="dataset")
    shutil.copyfile(cached, target)                        # copy out of the download cache


# ensure_snapshot: make sure data/raw/games_<label>.csv is the right file.
#   1. If the file is there and its fingerprint matches, use it (normal clone with Git LFS).
#   2. Otherwise try the release, then Hugging Face; each download is fingerprinted too.
#      A pointer file left by a clone without Git LFS fails step 1 and is replaced here.
#   3. If no source gives the right file, stop: running on a different file would give
#      different results without anyone noticing.
#   Returns the path of the verified file.
def ensure_snapshot(label: str):
    """Path of data/raw/games_<label>.csv, fetched if missing or wrong, always with the right SHA-256."""
    RAW.mkdir(parents=True, exist_ok=True)
    target = RAW / f"games_{label}.csv"
    if target.exists() and sha256_of(target) == SHA256[label]:
        print(f"snapshot {label.upper()}: data/raw copy verified")
        return target
    # try each source in order; the first copy with the right fingerprint is used
    for source, fetch in (("release raw-snapshots-v1", from_release),
                          ("Hugging Face", from_hugging_face)):
        try:
            fetch(label, target)
        except Exception as error:                          # no network, site down, ...
            print(f"snapshot {label.upper()}: {source} not reachable ({type(error).__name__})")
            continue
        if sha256_of(target) == SHA256[label]:
            print(f"snapshot {label.upper()}: downloaded from {source}, verified")
            return target
        print(f"snapshot {label.upper()}: copy from {source} has the wrong SHA-256")
    raise RuntimeError(f"snapshot {label.upper()}: no copy with SHA-256 {SHA256[label]} found")

In [ ]:
# 1.1c · Read only the 13 columns the analysis uses
# Why: the other 27 columns do not serve the questions; reading only the needed ones keeps the
# tables small and makes every column that is used visible here (list below).

# Raw column -> English snake_case name. Only these 12 columns (+ discount) of the 40 are read,
# each because a later step needs it:
#   AppID                  identifies a game; joins snapshot C to A (Step 3)
#   Name                   lets results and removed games be checked by eye (1.5, checks)
#   Release date           release-year band and the ">= 1 year on the store" filter
#   Price (+ Discount)     price tier; the discount recovers the list price from A's sale price
#   Supported languages    treatment (Simplified Chinese), other-languages count, Chinese-first
#                          signal 1, and the languages added between C and A (Step 3)
#   Full audio languages   voice-over signal, used only to test the filter (1.5b, 2d)
#   Positive, Negative     outcome: reach = positive + negative reviews
#   Developers             Chinese-first signal 2
#   Publishers             shown next to developers in the 1.5 check only; not a rule, because
#                          publishers are often regional partners
#   Genres                 Early Access flag only (Steam lists it as a genre)
#   Tags                   genre group (user tags, ordered by votes, are more specific than Genres)
# Not read, and why:
#   Estimated owners       ranges too wide (e.g. 0-20,000); review counts measure reach more finely
#   User score, Metacritic score, Score rank, Recommendations, Peak CCU, playtime (4 columns),
#   Achievements           other measures of quality or engagement; the question is about reach
#                          (how many players), measured one way, by reviews
#   Categories             single-/multiplayer and similar features; not held fixed in the groups
#   Required age, DLC count, Windows, Mac, Linux, Notes
#                          not related to translation or reach in this design
#   About the game, Reviews, Header image, Website, Support url, Support email, Metacritic url,
#   Screenshots, Movies    free text, links and media: no numbers to analyse, and text analysis
#                          is outside the project's scope
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Publishers": "publishers",
    "Genres": "genres",
    "Tags": "tags",
}
OPTIONAL_RAW_TO_CLEAN = {"Discount": "discount_pct"}   # only in snapshot A; "0" when missing

In [ ]:
# 1.1d · Repair the merged header of snapshot A (DATA ISSUE)
# Why: one header name covers two value columns, which shifts every later column by one; the
# name is split back before the file is read.

# read_header: read the column names and repair them (DATA ISSUE).
#   In snapshot A the header has 39 names over 40 value columns, because "Discount" and
#   "DLC count" were merged into one name. Splitting it back gives every column its own
#   name again; snapshot C has no such problem and passes through unchanged.
#   Returns the list of column names.
def read_header(raw_path) -> list[str]:
    """Header row of a raw file, with the merged name split back into two (DATA ISSUE)."""
    with open(raw_path, newline="", encoding="utf-8") as f:
        header = next(csv.reader(f))                       # first row only
    fixed = []
    for name in header:
        # "DiscountDLC count" -> ["Discount", "DLC count"]; any other name stays as it is
        fixed += ["Discount", "DLC count"] if name == "DiscountDLC count" else [name]
    return fixed

In [ ]:
# 1.1e · Read a snapshot as text, with English column names
# Why: reading every value as text and fixing the types in one place (1.2a) means no value is
# converted silently while reading; a check stops the load if columns still look shifted.

# load_snapshot: read one raw file into a table with only the columns the analysis uses.
#   - The repaired header is passed in, and the file's own first row is skipped.
#   - Every value is read as text (dtype=str): types are fixed later in one place, so no
#     value is silently converted while reading.
#   - Only an empty cell counts as missing; texts such as "NA" or "None" stay as text.
#   - A safety check stops the load if columns still look shifted.
#   Returns the table with English column names.
def load_snapshot(raw_path) -> pd.DataFrame:
    """Read one raw snapshot: needed columns only, English names, every value as text."""
    names = read_header(raw_path)
    missing = [c for c in RAW_TO_CLEAN if c not in names]
    if missing:
        raise ValueError(f"{raw_path.name}: columns not found: {missing}")
    optional = [c for c in OPTIONAL_RAW_TO_CLEAN if c in names]
    raw = pd.read_csv(
        raw_path, names=names, header=None, skiprows=1,    # use the repaired header
        usecols=list(RAW_TO_CLEAN) + optional,             # read only these columns
        dtype=str, keep_default_na=False, na_values=[""],  # text; only "" counts as missing
        engine="c",
    )
    # Stop if columns are still shifted: language cells must look like "[...]" lists.
    langs = raw["Supported languages"].dropna()
    share_list_like = langs.str.startswith("[").mean() if len(langs) else 0
    if share_list_like < 0.9:
        raise ValueError(f"{raw_path.name}: columns look shifted; check read_header()")
    out = raw[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
    for raw_name, clean_name in OPTIONAL_RAW_TO_CLEAN.items():
        # copy the column when the snapshot has it, otherwise fill with "0"
        out[clean_name] = raw[raw_name] if raw_name in raw.columns else "0"
    return out

In [ ]:
# 1.1f · Run: fetch (if needed), verify and read both snapshots
# Why: both snapshots go through the same functions, so they are loaded the same way.

raw_a = load_snapshot(ensure_snapshot("a"))
raw_c = load_snapshot(ensure_snapshot("c"))
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

**Check.** Snapshot A has 125,855 games and snapshot C 85,103, each with 13 columns (the 12
named columns and the discount). Snapshot A's first rows show a sale discount (65) next to
the price, which Step 1.2 uses to recover the list price.

## 1.2 Clean each snapshot and add the analysis columns

No game is removed here (except rows without a usable id); every later filter only uses the
flags added here.

**Data issues fixed**
- **Sale prices.** Snapshot A was taken during a Steam sale (Hades: 7.49 = 24.99 at -70%).
  `price` becomes the list price, `sale price / (1 - discount / 100)`, so the price tier
  describes the game, not the sale. Snapshot C has no discount column; its prices are used
  as listed.
- **Junk in list cells.** Language and tag cells contain leftovers from the store pages
  (`English[b]*[/b]`, HTML tags, line breaks); they are removed before parsing.
- **Ids and dates.** Rows without an id or with a repeated id are dropped (first kept);
  unreadable release dates become missing and are dropped by the filters.

**Columns added**
- **Outcome:** `total_reviews = positive + negative` (reach).
- **Treatment:** supports Simplified Chinese. Traditional-only games are flagged: they are
  neither "with Simplified Chinese" nor "without Chinese".
- **Confounders** (characteristics that affect reach and differ between games with and
  without Chinese; Steps 2-3 hold them fixed):
  - `genre_group`: the first of the game's user tags found in the genre map. Steam orders
    tags by votes, so this is the genre players use most. Tags such as "Indie" are skipped:
    they say nothing about how much text a game has.
  - `price_tier`: list price in 5 tiers, a stand-in for budget and studio size.
  - `year_band`: release year in 4 bands (time on the store).
  - `n_other_langs`, `n_other_langs_bin`: languages other than English and Chinese, in 3 bins.
- **Chinese-first games** (flag; removed in 1.3). The question is whether to translate a
  game into Chinese, so games whose original language is Chinese do not belong in the
  comparison: they have Chinese and Chinese players by default, and keeping them would mix
  "made for Chinese players" with "translated into Chinese". The data has no country or
  original-language column, so a game is flagged if either of two signals, both computed
  from the data only, is present:
  1. **Chinese text without English text.** A game sold internationally almost always has
     English, so a listing with Chinese and no English is made for Chinese-speaking players.
  2. **Every developer writes its name in Chinese characters** (e.g. 鬼谷工作室, the studio of
     Tale of Immortal, which ships English). Names that also contain kana are not counted:
     Japanese uses Chinese characters too, but kana exist only in Japanese (上海アリス幻樂団 is
     the Japanese studio of Touhou).

  Choices behind the two signals:
  - *No hand-made list of Chinese cities or companies.* Such a list holds only the names its
    author knows, and matching names is error-prone ("Paper Games" also matches the British
    studio White Paper Games).
  - *Voice-over languages are not used.* "Chinese voice-over without English voice-over" also
    describes many Japanese games dubbed in Chinese (NEKOPARA, STEINS;GATE); in the hand check
    of 1.5b only 1 of the 9 games flagged by voice-over alone was Chinese.
  - *Developers only, and all of them.* Publishers are often regional partners (方块游戏
    publishes Sekiro in Asia), and a game made elsewhere can list one co-developer with a
    Chinese name (ATRI -My Dear Moments-).
  - *What is missed.* A Chinese studio with an English name and an English release (Black
    Myth: Wukong) gives no signal in the data and stays in the sample. Step 2d shows that the
    result barely changes whether Chinese games are filtered or not.

**Encoding.** The three binned columns are saved as ordinal codes, the label's rank:

| Column | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

As text the labels sort in the wrong order ("10to20" before "5to10", "<=2017" after
"2022+"), which breaks tables and figure axes; codes sort correctly and can be used in
calculations that need an order. They are ranks, not amounts (tier 4 is not twice tier 2),
so they are not averaged or used as a plain number in a regression. `genre_group` is not
encoded: genres have no order, so a number would invent one.

In [ ]:
# 1.2a · Fix the types and the wrong single values (DATA ISSUES)
# Why: ids, review counts, prices and dates arrive as text and some are wrong: repeated or
# missing ids, sale prices instead of list prices (snapshot A was taken during a sale), dates
# in two formats. They are fixed before anything is computed from them.

# fix_types: give each column its real type and fix the wrong single values.
#   - app_id: text -> number; rows without a usable id or with a repeated id are dropped
#     (the first is kept), so each game appears once;
#   - positive / negative: text -> whole numbers; missing -> 0 (Steam shows none);
#   - price: list price = sale price / (1 - discount / 100); the discount is capped at 0-99%
#     because 100% would divide by zero; snapshot C has no discount column (0);
#   - release_date: "Aug 1, 2023" and "Aug 2023" both become dates; unreadable -> missing.
#   Returns a cleaned copy; the input table is not changed.
def fix_types(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Copy of one snapshot with real types, one row per id, list prices and dates."""
    df = trim.copy()
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")        # bad id -> NaN
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")        # DATA ISSUE: no / repeated id
    df["app_id"] = df["app_id"].astype("int64")
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # DATA ISSUE: sale prices in A -> list price = price / (1 - discount / 100)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # DATA ISSUE: dates come as "Aug 1, 2023" or "Aug 2023"; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days
    return df

In [ ]:
# 1.2b · Turn the list-like text cells into clean lists (DATA ISSUE)
# Why: languages, tags and genres are stored as text that looks like a list, in several shapes
# and with store-page junk; every rule below needs them as clean Python lists.

# DATA ISSUE: store-page leftovers in list cells: BBCode tags, HTML tags/entities, "*", line breaks
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


# parse_list: turn one list-like text cell into a clean Python list.
#   The raw cells come in several shapes: "['English', 'French']", "English,French", empty,
#   or with store-page junk such as "English[b]*[/b]". The steps:
#   1. missing cell -> empty list;
#   2. remove the junk;
#   3. "[...]" cells are read as a Python list; if that fails, or there are no brackets,
#      split on commas;
#   4. strip quotes and spaces, drop empty items and repeats, keep the original order
#      (the order matters for tags, which Steam sorts by votes).
def parse_list(value) -> list[str]:
    """List-like cell -> clean list of unique strings ("['A', 'B']", "A,B", junk, missing)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []                                          # missing cell
    text = _TAG_JUNK.sub("", str(value)).strip()           # remove the junk first
    if not text or text in {"[]", "nan"}:
        return []
    items = None
    if text.startswith("["):
        try:
            items = ast.literal_eval(text)                 # "['A', 'B']" -> ['A', 'B']
        except (ValueError, SyntaxError):
            text = text.strip("[]")                        # broken list: fall back to commas
    if items is None:
        items = text.split(",")                            # "A,B" -> ['A', 'B']
    cleaned = [str(i).strip().strip("'\"").strip() for i in items]
    seen, out = set(), []
    for item in cleaned:                                   # drop empties and duplicates,
        if item and item not in seen:                      # keep the original order
            seen.add(item)
            out.append(item)
    return out

In [ ]:
# 1.2c · Outcome, treatment and the other-languages confounder
# Why: reach is measured by the number of reviews (Steam does not publish sales); the
# treatment is Simplified Chinese support; the number of other languages is kept because
# games with Chinese are usually translated into many languages, which affects reach too.

# Languages: the treatment, and the language every Western game ships with.
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"


# add_outcome_and_languages: the outcome and every language column.
#   total_reviews = positive + negative (reach). From the language list: has Simplified /
#   Traditional Chinese / English, the number of OTHER languages (set difference: all minus
#   English and both Chinese), and all languages joined by "|" so Step 3 can compare the
#   lists of snapshots A and C.
#   Returns the same table with the new columns.
def add_outcome_and_languages(df: pd.DataFrame, langs: pd.Series) -> pd.DataFrame:
    """Add total_reviews and the language columns."""
    df["total_reviews"] = df["positive"] + df["negative"]
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))   # set difference
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))                   # "English|French|..."
    return df

In [ ]:
# 1.2d · Flag the Chinese-first games
# Why: a game whose original language is Chinese has Chinese and Chinese players by default,
# so it would mix "made for Chinese players" with "translated into Chinese". The data has no
# country column, so the flag is built from two signals computed from the data only (markdown
# above); the games are removed in 1.3.

# Legal suffixes that become separate items when a developer cell is split on commas.
COMPANY_SUFFIXES = ["inc", "ltd", "llc", "co", "corp", "limited", "gmbh", "plc", "sa", "srl", "co. ltd"]


_HAN = re.compile(r"[一-鿿]")       # a Chinese character (also used in Japanese kanji)
_KANA = re.compile(r"[぀-ヿ]")      # hiragana or katakana: only Japanese uses them


# split_developers: split a developer cell into one name per company.
#   Developers are separated by commas, but so are legal suffixes ("Studio X, Inc."). Parts
#   that are only a suffix (inc, ltd, ...) are dropped so they do not count as a developer.
#   Returns the list of names.
def split_developers(text: str) -> list[str]:
    """"A, B, Inc." -> ["A", "B"]: split on commas and drop legal suffixes."""
    parts = [p.strip().strip(".").strip() for p in str(text).split(",")]   # split on commas, trim spaces and dots
    suffixes = set(COMPANY_SUFFIXES)
    return [p for p in parts if p and p.lower().rstrip(".") not in suffixes]   # drop empty parts and suffixes


# all_match: do ALL developers of a game match a pattern?
#   "All", not "any", because a game made elsewhere can list one co-developer with a
#   Chinese name; "any" would remove it.
#   A game with no developer listed returns False (no evidence that it is Chinese).
def all_match(developers: list[str], pattern: re.Pattern) -> bool:
    """True if there is at least one developer and every developer matches `pattern`."""
    return bool(developers) and all(pattern.search(d) for d in developers)   # False for an empty list


# add_chinese_game_flags: one True/False column per signal, then the flag used by the filter.
#   cn_text      Chinese text without English text                        (signal 1, used)
#   cn_dev_name  every developer name has Chinese characters and no kana  (signal 2, used)
#   cn_dev_kana  every developer name has Chinese characters, some kana   (Japanese; not used)
#   cn_audio     Chinese voice-over without English voice-over            (not used)
#   is_cn_game = cn_text or cn_dev_name. The two signals that are not used are kept as
#   columns so that 1.5b and 2d can test what would change with them.
#   cn_rules names the used signals a game matched, and makers joins developers and
#   publishers, both for the check in 1.5.
def add_chinese_game_flags(df: pd.DataFrame, audio: pd.Series) -> pd.DataFrame:
    """Add the four signal columns, is_cn_game and the columns used to check it."""
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    developers = df["developers"].fillna("").astype(str)
    publishers = df["publishers"].fillna("").astype(str)
    dev_lists = developers.map(split_developers)
    # (Python's re via .map, not .str.contains: with pyarrow installed pandas uses Arrow's
    # regex engine, which rejects some of these patterns.)
    han_name = dev_lists.map(lambda d: all_match(d, _HAN))             # all developers: Chinese characters
    kana_name = developers.map(lambda text: bool(_KANA.search(text)))  # any kana in the developer cell
    has_zh_audio = audio.map(lambda x: bool(set(x) & chinese))
    has_en_audio = audio.map(lambda x: ENGLISH in x)

    df["cn_text"] = (df["has_zh_simplified"] | df["has_zh_traditional"]) & ~df["has_english"]
    df["cn_dev_name"] = han_name & ~kana_name
    df["cn_dev_kana"] = han_name & kana_name
    df["cn_audio"] = has_zh_audio & ~has_en_audio
    df["is_cn_game"] = df["cn_text"] | df["cn_dev_name"]               # the filter of 1.3

    used = df[["cn_text", "cn_dev_name"]]
    # names of the matched signals joined by ";", for the check in 1.5
    df["cn_rules"] = used.apply(lambda r: ";".join(r.index[r.to_numpy()]), axis=1)
    df["makers"] = developers + " | " + publishers
    return df

In [ ]:
# 1.2e · Flag Early Access and Traditional-Chinese-only games
# Why: Early Access games are unfinished and their reviews behave differently; a game with
# Traditional Chinese only is neither "with Simplified Chinese" nor "without Chinese". Both
# are flags here and are removed in 1.3, so the funnel can count them.

# add_filter_flags: the two True/False columns used by the filters in 1.3.
#   Steam lists "Early Access" as a genre, so it is found in the genres list.
def add_filter_flags(df: pd.DataFrame, genres: pd.Series) -> pd.DataFrame:
    """Add is_early_access and is_traditional_only."""
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]
    return df

In [ ]:
# 1.2f · Give each game one genre group
# Why: genre affects both the audience size and how much text a game has to translate. Six
# groups from the user tags keep the comparison groups large enough; the first matching tag
# wins because Steam orders tags by votes.

# Genre groups: user tag -> group; the first matching tag of a game wins.
TAG_TO_GENRE = {
    "Visual Novel": "Visual Novel",
    "RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Action RPG": "RPG", "Party-Based RPG": "RPG",
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "RTS": "Strategy",
    "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Action": "Action", "Shooter": "Action", "FPS": "Action", "Platformer": "Action",
    "Hack and Slash": "Action", "Fighting": "Action",
    "Puzzle": "Puzzle", "Hidden Object": "Puzzle",
    "Casual": "Casual",
}


# first_genre: give a game one genre group from its user tags.
#   Tags are ordered by how many players voted for them, so the first tag that appears in
#   TAG_TO_GENRE is the genre players most associate with the game. Tags outside the map
#   (e.g. "Indie", "Singleplayer") are skipped.
#   Returns the group name, or None if no tag matches (the game is then filtered out).
def first_genre(tags: list[str]) -> str | None:
    """Group of the first tag that appears in TAG_TO_GENRE; None if no tag matches."""
    for tag in tags:
        if tag in TAG_TO_GENRE:
            return TAG_TO_GENRE[tag]
    return None

In [ ]:
# 1.2g · Bin price, release year and other languages, and encode the bins
# Why: the comparison groups need a few levels, not exact values; the bins are saved as ranks
# 1, 2, 3, ... because as text they sort in the wrong order ("10to20" before "5to10").

# Bins; each label is saved as its position 1, 2, 3, ... in the list.
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]    # USD, list price
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]           # codes 1-5
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]    # codes 1-4
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]                              # codes 1-3


# to_code: bin a number and return the bin's rank.
#   pd.cut puts each value into the bins; labels=False gives the bin position 0, 1, 2, ...
#   and +1 turns it into 1, 2, 3, ... to match the code tables. Int64 (capital I) is the
#   integer type that can hold a missing value.
def to_code(values: pd.Series, bins: list) -> pd.Series:
    """Bin numbers and return the bin's rank 1, 2, 3, ... (missing stays missing)."""
    # labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


# add_codes: the three encoded confounders (codes in the table of the markdown above).
def add_codes(df: pd.DataFrame) -> pd.DataFrame:
    """Add price_tier, year_band and n_other_langs_bin as ranks."""
    df["price_tier"] = to_code(df["price"], PRICE_BINS)                # codes 1-5
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)      # codes 1-4
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)   # codes 1-3
    return df

In [ ]:
# 1.2h · Run: clean both snapshots the same way
# Why: one function chains 1.2a-1.2g, and both snapshots go through it, so a game is measured
# the same way in A and in C. No game is removed here except rows without a usable id.

# add_features: steps 1.2a-1.2g in order.
#   The list cells are parsed once (1.2b) and passed to the steps that need them.
#   Returns the cleaned table, one row per game.
def add_features(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Cleaned copy of one snapshot with all derived columns."""
    df = fix_types(trim, snapshot_date)                               # 1.2a
    langs = df["supported_languages"].map(parse_list)                 # 1.2b
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)
    df = add_outcome_and_languages(df, langs)                         # 1.2c
    df = add_chinese_game_flags(df, audio)                            # 1.2d
    df = add_filter_flags(df, genres)                                 # 1.2e
    df["genre_group"] = tags.map(first_genre)                         # 1.2f
    return add_codes(df)                                              # 1.2g


feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

**Check.** No game is removed yet: A still has 125,855 games and C 85,103.

## 1.3 Filter the rows

Applied in this order; each step is logged in a funnel (shown in 1.5).

| Filter | Why |
|---|---|
| drop Chinese-first games (either of the 2 signals) | Chinese is their original language, not a translation |
| drop Early Access | unfinished games; their reviews behave differently |
| keep >= 10 reviews | below that, reach is noise; not higher (e.g. 500), because filtering hard on the outcome itself would shrink the gap being measured |
| keep games released >= 1 year before the snapshot | every game had time to collect reviews |
| keep games with a genre group, price and release year | needed to place the game in a comparison group |
| `pcd_games_clean` only: drop Traditional-Chinese-only games | neither "with Simplified Chinese" nor "without Chinese" |

In [ ]:
# 1.3a · Apply filters in order and count what each removes
# Why: every filter is a named True/False rule applied one after another, so the funnel in
# 1.5 shows exactly how many games each rule removed.

# apply_filters: apply filters one after another and count what is left after each.
#   keep starts as all True; each filter turns some rows to False (&= keeps a row only if
#   it passed every filter so far). A missing value counts as failing. The counts after
#   each step form the funnel shown in 1.5.
#   Returns (rows that passed, funnel table).
def apply_filters(df: pd.DataFrame, steps: list[tuple[str, pd.Series]], table: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Keep rows passing every (name, mask) filter; return them and the funnel table."""
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)       # missing value = fails the filter
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)

In [ ]:
# 1.3b · The filters shared by both snapshots, and their thresholds
# Why: >= 10 reviews because below that reach is noise (not higher, which would filter on the
# outcome itself); >= 1 year on the store so every game had time to collect reviews. Both
# snapshots use this same list, so Step 1 and Step 3 measure games the same way.

MIN_REVIEWS = 10              # below that, reach is noise (see the table above)
MIN_AGE_DAYS = 365            # every game had a year to collect reviews


# base_filters: the filters both snapshots share, in the order they are applied.
#   Each filter is a (name, True = keep) pair; the name is what the funnel shows. Keeping
#   them in one function guarantees Step 1 and Step 3 filter A and C the same way.
def base_filters(df: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Filters used for both snapshots, as (name, True = keep) pairs, in order."""
    return [
        ("drop Chinese-first games (either of 2 signals)", ~df["is_cn_game"]),
        ("drop Early Access", ~df["is_early_access"]),
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

## 1.4 Build the cleaned dataset `pcd_games_clean`

One row per game of snapshot A that passed every filter: identifiers, confounders (raw
value and code), the treatment `has_zh` (1 = Simplified Chinese, 0 = no Chinese at all) and
the outcome. `log_total_reviews` is added because review counts are very skewed (a few hits
have millions), so figures read better on a log scale.

In [ ]:
# 1.4a · Build the cleaned dataset pcd_games_clean
# Why: one row per game of snapshot A that passed every filter, with only the columns the
# analysis uses; log reviews are added because review counts are very skewed.

# build_games_clean: produce the cleaned dataset pcd_games_clean from snapshot A.
#   Applies the shared filters plus "drop Traditional-only", turns the treatment into 1/0,
#   adds the log of reviews (rounded so the csv has the same bytes on every run), and keeps
#   only the 12 output columns, sorted by app_id so the row order never changes.
#   Returns (table, funnel).
def build_games_clean(feat_a: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cleaned snapshot A: one row per game after all filters, and its funnel."""
    steps = base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
    df, funnel = apply_filters(feat_a, steps, "games_clean")
    df["has_zh"] = df["has_zh_simplified"].astype(int)               # True/False -> 1/0
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6) # rounded: same bytes every run
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        df[col] = df[col].astype(int)                                # no missing left after filters
    cols = [
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# 1.4b · Run: build the cleaned dataset
# Why: snapshot A is the current catalogue, used for SQ1.

games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

**Check.** `games_clean` has 45,445 games, 14,701 of them with Simplified Chinese.

## 1.5 Check what the cleaning removed

The funnel shows how many games are left after each filter; no single step should remove
most games. The list of removed Chinese-first games (biggest first) is there to spot a wrongly
removed Western game by eye. Shown only, not saved.

In [ ]:
# 1.5a · Show the funnel and the biggest Chinese-first games removed
# Why: no single filter should remove most games, and a wrongly removed Western game is easy to
# spot among the biggest removed games. Shown only, not saved.

cn_cols = ["app_id", "name", "makers", "cn_rules", "has_english", "total_reviews"]
cn_excluded = (
    feat_a[feat_a["is_cn_game"] & (feat_a["total_reviews"] >= MIN_REVIEWS)][cn_cols]  # would pass the review filter
    .sort_values("total_reviews", ascending=False)                                   # biggest first
    .reset_index(drop=True)
)
print(f"{len(cn_excluded):,} Chinese-first games removed (>= {MIN_REVIEWS} reviews)")
display(funnel_a)
cn_excluded.head(20)

**Why?** The largest drop in the funnel is the review filter (110,107 → 50,160 games): most
games on Steam have fewer than 10 reviews. 2,030 games with at least 10 reviews are removed
as Chinese-first; the top of the list (鬼谷八荒 Tale of Immortal, The Scroll Of Taiwu, The
Hungry Lamb) is made in China, which is what the two signals are meant to catch.

### 1.5b Which signals can be trusted: hand check of a random sample

About 125,000 games cannot be checked one by one. Instead, 50 games were drawn at random
(`random_state=2010`, the same games on every run) from all games with at least 10 reviews
that show **any** of the four signals of 1.2d, and each was checked by hand against the
developer's name and store page. The cell below draws the sample and counts it by signal;
the last three columns are the result of the hand check.

| Signal (first that applies) | In the filter? | Games | Chinese-first | Not | Unclear |
|---|---|---:|---:|---:|---:|
| `cn_text`: Chinese text, no English text | yes | 25 | 25 | 0 | 0 |
| `cn_dev_name`: developer names in Chinese characters, no kana | yes | 12 | 11 | 1 | 0 |
| `cn_dev_kana`: developer names with kana | no | 4 | 0 | 4 | 0 |
| `cn_audio`: Chinese voice-over, no English voice-over | no | 9 | 1 | 4 | 4 |

- The two signals in the filter are right for 36 of 37 games. The one error is a Japanese
  studio whose name is written in kanji only (市松製化).
- Of the 25 `cn_text` games, 21 are made by Chinese-language studios (mainland China or
  Taiwan) and 4 are Chinese-only editions of foreign games; both are listings made for
  Chinese-speaking players, which is what the filter looks for.
- All 4 names with kana are Japanese studios, which is why kana names are not counted.
- Voice-over alone is right for 1 game of 9 (Japanese, Russian and Brazilian games are
  dubbed in Chinese too), which is why it is not used.
- A wrongly removed game only leaves the sample; it is not put on the wrong side of the
  comparison.

In [ ]:
# 1.5b · Draw the random sample that was checked by hand
# Why: the signals cannot be checked game by game, so a random sample shows how often each
# one is right. The sample is drawn from the games showing ANY of the four signals, so the
# two signals left out of the filter are tested as well; a fixed random_state draws the same
# 50 games on every run.

AUDIT_SIZE = 50               # games checked by hand
AUDIT_SEED = 2010             # fixed draw: the same sample on every run
SIGNALS = ["cn_text", "cn_dev_name", "cn_dev_kana", "cn_audio"]     # in the order of the table above
any_signal = feat_a[feat_a[SIGNALS].any(axis=1) & (feat_a["total_reviews"] >= MIN_REVIEWS)]
audit = any_signal.sample(n=AUDIT_SIZE, random_state=AUDIT_SEED).copy()
# idxmax on True/False columns gives the first column that is True: the first signal that applies
audit["signal"] = audit[SIGNALS].idxmax(axis=1)
print(f"{AUDIT_SIZE} of {len(any_signal):,} games with any signal, drawn with random_state={AUDIT_SEED}")
print(audit["signal"].value_counts().reindex(SIGNALS).to_string())
audit[["app_id", "name", "makers", "signal", "has_english", "total_reviews"]]

# Step 2 · SQ1 groups → `pcd_sq1_groups`

**Question.** How much larger is the reach of games that support Simplified Chinese, and in
which genres?

**Strategy: compare only similar games.** Comparing all games with and without Chinese
directly would mix the effect of Chinese with that of the confounders (bigger-budget games
are both more often translated and more reviewed). So every game of `pcd_games_clean` is put
in a **comparison group** of similar games, and games with and without Chinese are compared
**only inside the same group**: there the confounders are the same on both sides, so the
remaining gap is not explained by them. This is why the confounders need no other treatment
in the analysis.

**SQ1 group = genre x price tier x release-year band x number of other languages**

| Confounder | Why it is in the group |
|---|---|
| Genre | genres differ in audience size and in how much text a game has |
| Price tier | budget and studio size drive both translation and reach |
| Release-year band | older games have had more time to collect reviews; bands rather than single years, because single years leave only 47% of games in usable groups (79% with bands) |
| Other languages | the strongest: games with Chinese have a median of 7 other languages, games without it 0 (66% vs 9% have 5+); without it SQ1 would compare "localised in many languages" with "not localised" rather than measure Chinese |

**The table `pcd_sq1_groups`**, one row per group: number of games and **median** reviews
on each side (medians, because review counts are very skewed), and the gap
`pct_diff = median with Chinese / median without - 1`. A group is `eligible` only with >= 10
games on each side; a median of fewer games moves too much with one game. Result: 360
groups, 190 eligible, covering 79% of games (87% of games with Chinese).

In [ ]:
# 2a · Define the SQ1 comparison group
# Why: games with and without Chinese are compared only inside groups of similar games, so the
# confounders (genre, release-year band, price tier, other languages) are the same on both
# sides. The group name is written with labels so it can be read at a glance.

# code -> label, to write readable group names ("Action | <=2017 | lt5 | 5+")
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


# add_sq1_group: label each game with its SQ1 comparison group.
#   The group name joins the four confounders, e.g. "Action | <=2017 | lt5 | 5+". Codes are
#   turned back into labels so a group can be read at a glance.
def add_sq1_group(games: pd.DataFrame) -> pd.DataFrame:
    """Copy of games with group_id = genre | year band | price tier | other languages."""
    # assign returns a copy with the new column; the input table is not changed
    return games.assign(group_id=(
        games["genre_group"] + " | " + games["year_band"].map(YEAR_BAND_LABEL) + " | "
        + games["price_tier"].map(PRICE_LABEL) + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    ))

In [ ]:
# 2b · Compare games with and without Chinese inside each group
# Why: medians, because review counts are very skewed; a group counts (eligible) only with at
# least 10 games on each side, because a median of fewer games moves too much with one game.

MIN_GAMES_PER_SIDE = 10       # a group is eligible only with this many games per side (also Step 3)


# build_sq1_groups: compare games with and without Chinese inside each group.
#   1. groupby (group, has_zh): number of games and median reviews for each side;
#   2. unstack turns the two sides into columns, one row per group;
#   3. a side with no game gets 0 games and no median;
#   4. pct_diff = median with / median without - 1 (0.5 = 50% more reviews);
#   5. eligible = at least MIN_GAMES_PER_SIDE games on each side.
#   Returns one row per group, sorted by genre then codes.
def build_sq1_groups(games: pd.DataFrame) -> pd.DataFrame:
    """One row per group: games and median reviews with / without Chinese, gap, eligible."""
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")                    # count and median per group and side
        .unstack("has_zh")                                 # sides become columns: (stat, 0/1)
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)  # empty side -> 0
    out["pct_diff"] = (out["median_reviews_zh"] / out["median_reviews_no_zh"] - 1).round(6)
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)   # genre, then codes: old -> new, cheap -> expensive

In [ ]:
# 2c · Run: build the SQ1 group table
# Why: one row per group is the table SQ1 is answered from.

sq1_groups = build_sq1_groups(add_sq1_group(games_clean))
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

**Check.** 360 groups, 190 of them eligible.

## 2d Does the result depend on how Chinese-first games are filtered?

The filter of 1.2d cannot be perfect: a Chinese studio with an English name and an English
release (Black Myth: Wukong) is not detected. So SQ1 is computed again under other filters,
from none at all to a stricter one, and without the biggest hits. Each row runs Step 1.3-1.4
and Step 2 again on a temporary copy; `games_clean` and `sq1_groups` are not changed and
nothing is saved.

In [ ]:
# 2d · Compute SQ1 again under other filters
# Why: if the result were driven by Chinese games the filter misses, or by games it removes
# wrongly, it would move when the filter changes. "No filter" keeps every Chinese-first game;
# "stricter" also removes the games flagged by the two signals that were left out; the last
# row drops the 1% most-reviewed games, where the undetected Chinese hits are.

TOP_SHARE = 0.01              # share of most-reviewed games dropped in the last row


# sq1_summary: Steps 1.3-1.4 and 2 again for one filter, on a copy of feat_a.
#   assign() returns a copy with another is_cn_game column, so feat_a itself is not changed;
#   the results go into local variables, so games_clean and sq1_groups are not changed.
#   Returns one row of the table: games kept, eligible groups, median gap of eligible groups.
def sq1_summary(label: str, is_cn_game: pd.Series, drop_top: float = 0.0) -> dict:
    """SQ1 under the given Chinese-first flag, optionally without the most-reviewed games."""
    games, _ = build_games_clean(feat_a.assign(is_cn_game=is_cn_game))
    if drop_top:
        games = games[games["total_reviews"] <= games["total_reviews"].quantile(1 - drop_top)]
    groups = build_sq1_groups(add_sq1_group(games))
    eligible = groups[groups["eligible"]]
    return {"filter": label, "games": len(games), "eligible_groups": len(eligible),
            "median_pct_diff": round(float(eligible["pct_diff"].median()), 3)}


chosen = feat_a["cn_text"] | feat_a["cn_dev_name"]
stricter = chosen | feat_a["cn_dev_kana"] | feat_a["cn_audio"]
sensitivity = pd.DataFrame([
    sq1_summary("no filter", chosen & False),
    sq1_summary("chosen: text or developer name", chosen),
    sq1_summary("stricter: + kana names and voice-over", stricter),
    sq1_summary("chosen, without the 1% most-reviewed games", chosen, drop_top=TOP_SHARE),
])
sensitivity

**Why?** With no filter at all, games with Chinese have a median of +125% more reviews than
similar games without it; with the chosen filter +120%; with the stricter filter +117%; and
without the 1% most-reviewed games, which include the undetected Chinese hits, still +112%.
The gap stays large and of the same size in every row, so the answer to SQ1 does not depend
on which Chinese-first games the filter catches or misses.

# Step 3 · SQ2 groups → `pcd_sq2_groups`

**Question.** Does adding Simplified Chinese after launch still increase reach, and in
which genres?

**Strategy: follow the same games over time.** Take the games that had no Chinese in
January 2024 (snapshot C) and are still on Steam in June 2026 (snapshot A), and measure how
much their reviews grew. Games that added Chinese in between (**treated**) are compared with
similar games that added nothing (**control**). The control arm is needed because every game
gains reviews over time anyway; the uplift is the extra growth of the treated games.

**3.1 The panel** (one row per game, the input of the groups; not saved):
- start from cleaned snapshot C with the same filters as Step 1 (so the starting point is
  measured the same way), keep games with no Chinese at all in C and still present in A;
- game characteristics are taken from C, because what a game looked like **before** adding
  Chinese is what is held fixed;
- drop games with 0 reviews in A (delisted or missing data: a growth of -100% is not real);
- outcome `growth = (reviews in A - reviews in C) / reviews in C`;
- `arm`: `treated` = added Simplified Chinese; `control` = added neither Chinese nor any
  major language (German, French, Spanish, Japanese, Russian, Korean, Portuguese-Brazil),
  because a game that added German and French at the same time had a localisation push of
  its own and is not a clean "did nothing"; `excluded` = added another major language only.

**3.2 SQ2 group = genre x release-year band.** Only 510 games added Chinese, so the group
can hold only the confounders that matter most for **growth**:

| Variable | In the group? | Why |
|---|---|---|
| Genre | yes | needed to answer per genre |
| Release-year band | yes | the strongest: among control games, median growth is +12% for games from <=2017 and +77% for 2022+, and treated games are about 2 years newer; without it the effect of Chinese would be overstated |
| Price tier | no | once the year band is fixed, growth is about the same in every tier except > \$20; genre x price x year leaves only 17 usable groups, 54% of treated games and 3 genres (no RPG, Strategy or Visual Novel) |
| Other languages | no | treated and control games have almost the same mix (0 / 1-4 / 5+: 62/25/13% vs 64/26/10%) |

Result: 24 groups, 16 eligible, covering 90% of treated games and all 6 genres.

**The table `pcd_sq2_groups`**, built as in Step 2 but with growth: number of treated and
control games, their median growth, and
`uplift_pts = median growth treated - median growth control` (0.1 = +10 growth points);
`eligible` with >= 10 games on each side.

In [ ]:
# 3a · Treated, control or excluded: what a game added between C and A
# Why: treated = added Simplified Chinese. Control = added neither Chinese nor a major language,
# because a game that added German and French at the same time had a localisation push of its
# own and is not a clean "did nothing". Excluded = added another major language only.

MAJOR_LANGS = [               # a control game must not have added any of these
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]


# label_arm: compare each game's language lists in A and C and give it its arm.
#   Languages in A but not in C were added after January 2024 (set difference per game).
#   np.select gives the label of the first condition that is True, otherwise the default.
#   Returns one label per game, in the order of the inputs.
def label_arm(a: pd.DataFrame, c: pd.DataFrame) -> np.ndarray:
    """treated / control / excluded for games without Chinese in C."""
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    added_zh = a["has_zh_simplified"].to_numpy()           # had no Chinese in C, has it in A
    added_major = np.array([bool(n & set(MAJOR_LANGS)) for n in new_langs])
    # np.select: first condition that is True gives the label, otherwise the default
    return np.select([added_zh, ~added_zh & ~added_major], ["treated", "control"], default="excluded")

In [ ]:
# 3b · Build the SQ2 panel: the same games followed from C to A
# Why: start from games with no Chinese in January 2024, filtered as in Step 1, and still on
# Steam in June 2026; their characteristics are taken from C (before any change), and games
# with 0 reviews in A (delisted) are dropped because a -100% growth is not real.

# build_sq2_panel: follow the games that had no Chinese in C into A.
#   1. filter C exactly as in Step 1, then keep games with no Chinese that still exist in A;
#   2. take each game's characteristics from C (before any change) and its reviews from both;
#   3. drop games with 0 reviews in A (delisted) and compute growth;
#   4. give each game its arm with label_arm (3a): treated / control / excluded.
#   Returns (panel, funnel).
def build_sq2_panel(feat_a: pd.DataFrame, feat_c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per game of C (no Chinese, filtered as in Step 1) also present in A; and its funnel."""
    steps = base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_panel")
    c = c.set_index("app_id")
    a = feat_a.set_index("app_id").loc[c.index]            # same games, same order, from A

    p = pd.DataFrame(index=c.index)
    for col in ["name", "genre_group", "release_year", "year_band", "price_tier"]:
        p[col] = c[col]                                    # characteristics BEFORE: from C
    p["price_c"] = c["price"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    p = p[p["reviews_a"] > 0].copy()                       # drop delisted games (0 reviews in A)
    a, c = a.loc[p.index], c.loc[p.index]
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)

    p["arm"] = label_arm(a, c)                             # 3a: treated / control / excluded
    for col in ["year_band", "price_tier"]:
        p[col] = p[col].astype(int)                        # no missing left after filters
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_panel", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_panel", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_panel", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    cols = ["name", "genre_group", "release_year", "year_band", "price_c", "price_tier",
            "reviews_c", "reviews_a", "growth", "arm"]
    return p[cols].reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# 3c · Run: build the panel and show its funnel
# Why: the funnel shows how many games each step keeps.

sq2_panel, funnel_c = build_sq2_panel(feat_a, feat_c)
print(f"sq2_panel: {len(sq2_panel):,} games")
print(sq2_panel["arm"].value_counts().to_string())
display(funnel_c)

**Check.** The panel has 21,590 games: 510 treated, 20,839 control and 241 excluded.

In [ ]:
# 3d · Compare treated and control growth inside each SQ2 group
# Why: group = genre x release-year band, the confounders that matter most for growth (see the
# table above); medians and the >= 10 games per side rule as in Step 2.

# add_sq2_group: label each panel game with its SQ2 group, e.g. "Action | 2022+".
def add_sq2_group(panel: pd.DataFrame) -> pd.DataFrame:
    """Copy of the panel with group_id_sq2 = genre | year band."""
    # assign returns a copy with the new column; map turns the year code back into its label
    return panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))


# build_sq2_groups: compare the growth of treated and control games inside each group.
#   Same steps as build_sq1_groups, on growth instead of reviews; "excluded" games are left
#   out. uplift_pts = median growth treated - median growth control.
#   Returns one row per group.
def build_sq2_groups(panel: pd.DataFrame) -> pd.DataFrame:
    """One row per group: treated and control games, their median growth, uplift, eligible."""
    keys = ["group_id_sq2", "genre_group", "year_band"]
    used = panel[panel["arm"].isin(["treated", "control"])]          # "excluded" is not compared
    # count and median growth per group and arm; unstack makes treated / control columns
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)  # empty arm -> 0
    out["uplift_pts"] = (out["median_growth_treated"] - out["median_growth_control"]).round(6)
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)          # genre, then old -> new

In [ ]:
# 3e · Run: build the SQ2 group table
# Why: one row per group is the table SQ2 is answered from.

sq2_groups = build_sq2_groups(add_sq2_group(sq2_panel))
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

**Check.** 24 groups, 16 of them eligible.

# Save and check

The three tables are written whole as csv (for code) and xlsx (to view). An xlsx file is a zip
archive that records the time it was written, inside the file and on every zip entry; both
are set to a fixed date, so its content depends only on the table. The last lines printed are
fingerprints of the saved tables: the same values on every machine (for an xlsx, of its
unzipped content, since zip compression differs between zlib builds). The checks then stop the
notebook with an error if anything is wrong, including any of the numbers in the **Check**
notes above.

In [ ]:
# Save a · Write the three tables as csv and xlsx
# Why: csv to read in code, xlsx to open and look at; the xlsx files get a fixed date so their
# content depends only on the table.

tables = {
    "games_clean": games_clean,   # Step 1: cleaned data, one row per game
    "sq1_groups": sq1_groups,     # Step 2: one row per SQ1 group
    "sq2_groups": sq2_groups,     # Step 3: one row per SQ2 group
}
FIXED_DATE = datetime(2026, 1, 1)                     # written into every xlsx instead of "now"


# write_xlsx: save a table as xlsx without the time it was written.
#   An xlsx file is a zip archive of XML files. pandas/openpyxl write it to a draft file;
#   the draft is then copied entry by entry into the final file, with the save time inside
#   docProps/core.xml and the timestamp of every zip entry replaced by FIXED_DATE. The
#   content then depends only on the table, so a rerun does not show a change in git.
def write_xlsx(table: pd.DataFrame, path, sheet: str) -> None:
    """Write an xlsx whose bytes depend only on the table: fixed dates inside and on the zip entries."""
    draft = path.with_name(path.stem + "_draft.xlsx")      # temporary file next to the final one
    with pd.ExcelWriter(draft, engine="openpyxl") as writer:
        table.to_excel(writer, index=False, sheet_name=sheet)
    stamp = FIXED_DATE.strftime("%Y-%m-%dT%H:%M:%SZ").encode()   # "2026-01-01T00:00:00Z" as bytes
    with zipfile.ZipFile(draft) as src, zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as dst:
        for item in src.infolist():
            data = src.read(item.filename)
            if item.filename == "docProps/core.xml":       # openpyxl writes the save time here
                data = re.sub(rb"(<dcterms:(?:created|modified)[^>]*>)[^<]*", rb"\g<1>" + stamp, data)
            entry = zipfile.ZipInfo(item.filename, date_time=FIXED_DATE.timetuple()[:6])
            entry.compress_type = zipfile.ZIP_DEFLATED
            dst.writestr(entry, data)                      # same entries, fixed timestamps
    draft.unlink()                                         # delete the draft


PROCESSED_CSV.mkdir(parents=True, exist_ok=True)
PROCESSED_XLSX.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV / f"{PREFIX}{name}.csv", index=False)
    write_xlsx(table, PROCESSED_XLSX / f"{PREFIX}{name}.xlsx", sheet=name[:31])   # sheet names max 31 chars
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

In [ ]:
# Save b · Check the structure of the tables and the saved files
# Why: catches broken ids, codes out of range, non-English column names, or an xlsx that does
# not hold the same table as the csv.

# structure: unique ids, codes in range, English snake_case column names
assert games_clean["app_id"].is_unique and sq2_panel["app_id"].is_unique
assert set(games_clean["has_zh"]) <= {0, 1}
assert set(sq2_panel["arm"]) <= {"treated", "control", "excluded"}
assert games_clean["price_tier"].between(1, 5).all() and games_clean["year_band"].between(1, 4).all()
assert games_clean["n_other_langs_bin"].between(1, 3).all()
for name, table in tables.items():
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in table.columns), name

# saved files: the xlsx holds the same table as the csv
for name, table in tables.items():
    back = pd.read_excel(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx")
    assert back.shape == table.shape and list(back.columns) == list(table.columns), name

In [ ]:
# Save c · Check the data issues, the rules and the groups on known cases
# Why: known games show whether the fixes and rules behave as intended; the group checks show
# that every game is in exactly one group and eligible groups really have 10 games per side.

# known games: data issues and rules behave as intended
by_name = games_clean.set_index("name")
assert by_name.loc["Hades", "price"] > 20                             # list price, not the 7.49 sale price
assert by_name.loc["Hades", "price_tier"] == PRICE_LABELS.index("gt20") + 1
for name in ["Cyberpunk 2077", "Far Cry® 5", "Core Keeper", "Hollow Knight",   # Western games
             "Ether One", "STEINS;GATE"]:              # a name like a Chinese company; Japanese, dubbed in Chinese
    assert name in by_name.index, f"{name} should be kept"
for name in ["鬼谷八荒 Tale of Immortal", "The Scroll Of Taiwu", "Sword and Fairy 7"]:
    assert name in set(cn_excluded["name"]), f"{name} should be removed"  # signal 2, signal 1, both
assert sq2_panel["growth"].min() > -1                                 # no delisted game left

# group tables: every game in exactly one group, eligible groups really have >= 10 games per side
assert sq1_groups[["n_zh", "n_no_zh"]].to_numpy().sum() == len(games_clean)
assert sq2_groups[["n_treated", "n_control"]].to_numpy().sum() == (sq2_panel["arm"] != "excluded").sum()
assert (sq1_groups.loc[sq1_groups["eligible"], ["n_zh", "n_no_zh"]] >= MIN_GAMES_PER_SIDE).all().all()
assert (sq2_groups.loc[sq2_groups["eligible"], ["n_treated", "n_control"]] >= MIN_GAMES_PER_SIDE).all().all()

In [ ]:
# Save d · Check the numbers reported in the notes and the README
# Why: the markdown and the README quote these numbers; if a rerun gives others, the notebook
# stops instead of leaving the text out of date.

# the run reproduces the numbers given in the Check notes and in the README
assert (len(raw_a), len(raw_c)) == (125_855, 85_103)
assert (len(games_clean), int(games_clean["has_zh"].sum())) == (45_445, 14_701)
assert len(cn_excluded) == 2_030
assert (len(sq1_groups), int(sq1_groups["eligible"].sum())) == (360, 190)
assert sq2_panel["arm"].value_counts().to_dict() == {"control": 20_839, "treated": 510, "excluded": 241}
assert (len(sq2_groups), int(sq2_groups["eligible"].sum())) == (24, 16)
# the sensitivity check: its "chosen" row is the main result, and the reported gaps
main_gap = round(float(sq1_groups.loc[sq1_groups["eligible"], "pct_diff"].median()), 3)
assert sensitivity.loc[1, "median_pct_diff"] == main_gap and sensitivity.loc[1, "games"] == len(games_clean)
assert sensitivity["median_pct_diff"].tolist() == [1.254, 1.2, 1.174, 1.121]
assert audit["signal"].value_counts().reindex(SIGNALS).tolist() == [25, 12, 4, 9]
print("all checks passed")

In [ ]:
# Save e · Print the fingerprints of the saved tables
# Why: two runs on any machine must print the same values, which is quick to compare by eye.

# Fingerprints of the saved tables: two runs on any machine print the same values.
# content_fingerprint: fingerprint an output file by what it contains.
#   A csv is fingerprinted as it is. An xlsx is unzipped first and its entries are
#   fingerprinted in name order, because zip compression can differ between machines while
#   the content does not.
def content_fingerprint(path) -> str:
    """SHA-256 of a csv file, or of the unzipped entries of an xlsx file."""
    if path.suffix != ".xlsx":
        return sha256_of(path)
    digest = hashlib.sha256()
    with zipfile.ZipFile(path) as archive:                 # open the xlsx as the zip it is
        for name in sorted(archive.namelist()):
            digest.update(name.encode() + archive.read(name))   # entry name + its unzipped bytes
    return digest.hexdigest()


for path in sorted([*PROCESSED_CSV.glob(f"{PREFIX}*.csv"), *PROCESSED_XLSX.glob(f"{PREFIX}*.xlsx")]):
    print(f"{content_fingerprint(path)[:16]}  {path.relative_to(ROOT)}")

# Part B · Analysis: figures and insights

Steps 1-3 did the data wrangling; Part B only reads the three saved tables, draws the figures
(saved in `report/figures/`) and states the insights.

**Run Part B on its own.** Run Setup a-c at the top, then go on from cell B a below. The tables must exist: they do after Part A has run once (and the committed copies are
in the repository).

| Section | Reads |
| :--- | :--- |
| B1 EDA | `pcd_games_clean` |
| B2 SQ1 | `pcd_sq1_groups` |
| B3 SQ2 | `pcd_sq2_groups` |
| B4 SQ3 | `pcd_sq1_groups` and `pcd_sq2_groups`, per genre |

In [ ]:
# B a · Load the saved tables and prepare the figures folder
# Why: Part B reads only the saved tables, never variables of Part A, so it can run on its own.

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROCESSED_CSV = ROOT / "data" / "processed" / "csv"
PREFIX = "pcd_"
FIGURES = ROOT / "report" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


# load: read one saved output table for the analysis.
#   Part B reads only the saved csv files, never variables of Part A, so it can run on its
#   own; a clear message explains what to do when a table has not been written yet.
def load(name: str) -> pd.DataFrame:
    """Read one saved output table; stop with a clear message if Part A has not written it."""
    path = PROCESSED_CSV / f"{PREFIX}{name}.csv"           # e.g. data/processed/csv/pcd_games_clean.csv
    if not path.exists():
        raise FileNotFoundError(f"{path.relative_to(ROOT)} is missing: run Part A first.")
    return pd.read_csv(path)


games_clean = load("games_clean")
sq1_groups = load("sq1_groups")
sq2_groups = load("sq2_groups")
print(f"games_clean {len(games_clean):,} rows | sq1_groups {len(sq1_groups)} | sq2_groups {len(sq2_groups)}")

## B1. EDA

## B2. SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

## B3. SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

## B4. SQ3: for each genre, translate at launch, later, or not a priority?